Subject: Issue with Recent Order

Hi Customer Support,

I recently ordered a laptop from your website, but I received a damaged product. Could you please arrange for a replacement or refund my money? Additionally, I was charged twice for the same order. Please look into this matter urgently.

Thank you,
John Doe


In [2]:
import spacy
import re

# Load Spacy model
nlp = spacy.load("en_core_web_sm")

# Sample email text
email_text = """
Hi Customer Support,

I recently ordered a laptop from your website, but I received a damaged product. Could you please arrange for a replacement or refund my money? Additionally, I was charged twice for the same order. Please look into this matter urgently.

Thank you,
John Doe
"""


# Preprocessing
def preprocess_text(text):
    text = text.lower()  # Lowercase
    text = re.sub(r"\n", " ", text)  # Replace newlines with space
    text = re.sub(r"[^\w\s]", "", text)  # Remove punctuation
    return text


processed_text = preprocess_text(email_text)
print(processed_text)

 hi customer support  i recently ordered a laptop from your website but i received a damaged product could you please arrange for a replacement or refund my money additionally i was charged twice for the same order please look into this matter urgently  thank you john doe 


In [4]:
import benepar
import spacy

# Load Spacy and Benepar models
nlp = spacy.load("en_core_web_sm")
benepar.download("benepar_en3")
nlp.add_pipe("benepar", config={"model": "benepar_en3"})

# Parse the text
doc = nlp(processed_text)

# Constituency Parsing
for sent in doc.sents:
    print(sent._.parse_string)

# Dependency Parsing
for token in doc:
    print(f"{token.text} -> {token.head.text} ({token.dep_})")

[nltk_data] Downloading package benepar_en3 to
[nltk_data]     /Users/amitpandey/nltk_data...
[nltk_data]   Unzipping models/benepar_en3.zip.
/opt/anaconda3/lib/python3.11/site-packages/benepar/parse_chart.py:169: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file.

(S (UH  ) (RB hi) (NN customer) (NN support) (NNP  ) (S (NP (PRP i)) (ADVP (RB recently)) (VP (VBD ordered) (NP (DT a) (NN laptop)) (PP (IN from) (NP (PRP$ your) (NN website))))) (CC but) (S (NP (PRP i)) (VP (VP (VBD received) (NP (DT a) (VBN damaged) (NN product))) (MD could) (NP (PRP you)) (VP (ADVP (VB please)) (VP (VP (VB arrange) (PP (IN for) (NP (DT a) (NN replacement)))) (CC or) (VP (VB refund) (NP (PRP$ my) (NN money))))))) (ADVP (RB additionally)) (NP (PRP i)) (VP (VP (VBD was) (VP (VBN charged) (ADVP (RB twice)) (PP (IN for) (NP (DT the) (JJ same) (NN order))))) (ADVP (RB please)) (VP (VB look) (PP (IN into) (NP (DT this) (NN matter))) (ADVP (RB urgently)) (PP (VB  ) (S (VP (VB thank) (NP (NP (PRP you)) (NP (NNP john) (NNP doe)))))))))
  -> hi (dep)
hi -> support (amod)
customer -> support (compound)
support -> ordered (npadvmod)
  -> support (dep)
i -> ordered (nsubj)
recently -> ordered (advmod)
ordered -> ordered (ROOT)
a -> laptop (det)
laptop -> ordered (dobj)
from -> or

/opt/anaconda3/lib/python3.11/site-packages/torch/distributions/distribution.py:55: UserWarning: <class 'torch_struct.distributions.TreeCRF'> does not define `arg_constraints`. Please set `arg_constraints = {}` or initialize the distribution with `validate_args=False` to turn off validation.
  warnings.warn(


In [5]:
def extract_features(doc):
    features = {"requests": [], "actions": [], "entities": []}

    for sent in doc.sents:
        for token in sent:
            if token.dep_ == "dobj":
                features["requests"].append(token.text)
            elif token.dep_ == "ROOT" and token.pos_ == "VERB":
                features["actions"].append(token.text)
            if token.ent_type_:
                features["entities"].append((token.text, token.ent_type_))

    return features


features = extract_features(doc)
print(features)

{'requests': ['laptop', 'product', 'money', 'you'], 'actions': ['ordered'], 'entities': [('john', 'PERSON')]}


In [6]:
def classify_email(features):
    if "refund" in features["actions"]:
        return "Refund Request"
    elif "replacement" in features["requests"]:
        return "Replacement Request"
    elif "charged" in features["actions"]:
        return "Billing Issue"
    else:
        return "General Inquiry"


category = classify_email(features)
print(f"Email Category: {category}")

Email Category: General Inquiry


In [8]:
def extract_actionable_items(features):
    actions = []

    if "refund" in features["actions"]:
        actions.append("Process refund for the customer.")
    if "replacement" in features["requests"]:
        actions.append("Arrange a replacement for the damaged product.")
    if "charged" in features["actions"]:
        actions.append("Investigate the double charge issue.")
    if "ordered" in features["actions"]:
        actions.append("Person has problems with his order.")

    return actions


actionable_items = extract_actionable_items(features)
print("Actionable Items:")
for item in actionable_items:
    print(f"- {item}")

Actionable Items:
- Person has problems with his order.


Actionable Items:
- Process refund for the customer.
- Arrange a replacement for the damaged product.
- Investigate the double charge issue.